In [0]:
import os
import mlflow

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/1")

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# Read test data and normalize (model was trained on normalized features)
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Filter to latest 3 weeks
max_date = test_norm.agg(F.max("timeStamp")).collect()[0][0]
test_recent = test_norm.filter(F.col("timeStamp") >= F.date_sub(F.lit(max_date), 21))

# Run predictions
predictions = model.transform(test_recent)

# Join with original table to get actual fault state and deviceId
original = spark.read.table("teams.sensorx.df_sx_800_with_delta").select("timeStamp", "payload_fault_state", "properties_deviceId")

pred_with_fault = predictions.select(
    "timeStamp", "label", "prediction"
).join(original, on="timeStamp", how="left") \
 .withColumn("date", F.to_date("timeStamp"))

# One prediction per day per device
daily_results = pred_with_fault.groupBy("date", "properties_deviceId").agg(
    F.max("prediction").cast("int").alias("daily_prediction"),
    F.max(F.col("payload_fault_state").cast("int")).alias("actual_fault_state"),
    F.max("label").alias("horizon_failure"),
    F.count("*").alias("readings_count")
).orderBy("date", "properties_deviceId")

display(daily_results)

In [0]:
from pyspark.sql import Row

data = [
    Row(Trial=1,  AUC=0.6525), #Logistic Regression 1, 5.mars
    Row(Trial=2,  AUC=0.6251), #Random Forest 1, 5.mars
    Row(Trial=3,  AUC=0.5383), #Gradient Boosted Trees 1, 5.mars
    Row(Trial=4,  AUC=0.9115), #Logistic Regression 2, 8.mars
    Row(Trial=5,  AUC=0.9139), #Logistic Regression 3, 8.mars
    Row(Trial=6,  AUC=0.9147), #Gradient Boosted Trees 2, 8.mars
    Row(Trial=7,  AUC=0.9114), #Logistic Regression 4, 12.mars
    Row(Trial=8,  AUC=0.9005), #Logistic Regression 5, 12.mars
    Row(Trial=9,  AUC=0.9237), #Logistic Regression 6, 12.mars
    Row(Trial=10, AUC=0.9350), #Logistic Regression 7, 12.mars
    Row(Trial=11, AUC=0.9411), #Random Forest 2, 12.mars
    Row(Trial=12, AUC=0.9156), #Gradient Boosted Trees 3, 12.mars
    Row(Trial=13, AUC=0.9240), #Random Forest 3, 15.mars
    Row(Trial=14, AUC=0.9360), #Gradient Boosted Trees 5, 17.mars
    Row(Trial=15, AUC=0.9379), #Multilayer Perceptron 1, 23.mars
    Row(Trial=16, AUC=0.9238), #Multilayer Perceptron 2, 29.mars
    Row(Trial=17, AUC=0.9136), #Multilayer Perceptron 3, 29.mars
    Row(Trial=18, AUC=0.9137), #Multilayer Perceptron 4, 29.mars
    Row(Trial=19, AUC=0.9034), #Multilayer Perceptron 5, 29.mars
    Row(Trial=20, AUC=0.8837), #Multilayer Perceptron 6, 29.mars
    Row(Trial=21, AUC=0.8847), #Multilayer Perceptron 7, 29.mars
    Row(Trial=22, AUC=0.9314), #Multilayer Perceptron 8, 29.mars
    Row(Trial=23, AUC=0.9452), #Multilayer Perceptron 9, 2.apríl
]

df_auc = spark.createDataFrame(data)
display(df_auc)

Databricks visualization. Run in Databricks to view.

In [0]:
import plotly.graph_objects as go



rows = df_auc.orderBy("Trial").collect()
trials = [r.Trial for r in rows]
aucs = [r.AUC for r in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=trials, y=aucs, mode="lines+markers", name="AUC", line=dict(color="#919191")))

# Lines with information about where LR and MLP stop/start
fig.add_vline(x=10, line_dash="dash", line_color="#BF7080",
              annotation_text="LR testing stops", annotation_position="top left")
fig.add_vline(x=16, line_dash="dash", line_color="#99DDB4",
              annotation_text="MLP testing starts", annotation_position="top left")

fig.update_layout(
    title="AUC vs Trials",
    xaxis_title="Trial",
    yaxis_title="AUC",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig.show()
